# 04 · 按控制步定位第一次可观察错误
预计20–30分钟。区分日志/画面事实、原因假设和改进。夹爪开口以米表示，夹爪命令是[-1,1]的无量纲数，分轴展示。若学生主模型全部成功，使用明确标注的真实教师失败；学生评测结果保持原样。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
GROUP = os.environ.get("LAB_GROUP", "group_01")  # 教师为每组分配；只改自己的组
RUN = workspace(GROUP)
from manipulation_lab.teaching import read_json, markdown_table, paired_cases
EXPERIMENT = stage_path(RUN, "experiment")
experiment = read_json(EXPERIMENT / "experiment.json")
assert experiment["complete"], "先完成 Notebook 02 的两组对照实验"
primary = experiment["variants"][experiment["primary_variant"]]
TRAINING = EXPERIMENT / primary["training"]
EVALUATION = EXPERIMENT / primary["evaluation"]


In [ ]:
FAILURE_EVALUATION = EVALUATION
results = read_json(FAILURE_EVALUATION / "summary.json")
failures = [r for r in results["episodes"] if not r["stable_success"] and not r["error"] and r["video"]]
if not failures:
    reference = Path(os.environ.get("LAB_REFERENCE_ROOT", ROOT / "checkpoints"))
    FAILURE_EVALUATION = reference / "reference_evaluation"
    results = read_json(FAILURE_EVALUATION / "summary.json")
    assert results["policy_role"] == "instructor_reference"
    failures = [r for r in results["episodes"] if not r["stable_success"] and not r["error"] and r["video"]]
    if not failures:
        raise RuntimeError("没有可分析的真实失败：请教师准备；不能伪造失败。")
    print("分析来源：教师参考；学生结果未替换。")
display(Markdown(markdown_table(failures, ["episode_id", "source_episode", "steps", "environment_success", "stable_success"])))
EPISODE = failures[0]["episode_id"]  # 可选表内其他失败
selected = next(r for r in failures if r["episode_id"] == EPISODE)
print("Selected:", EPISODE, "可选控制步范围 0..", selected["steps"])


先看完整录像，再修改下面的 KEYFRAME_STEPS。FIRST_ERROR_STEP 由你填写第一次可观察错误的控制步，尚不能判断时保留 None。控制步0是动作前状态；动作a[t-1]作用后得到状态s[t]。视频每4步采样一次，选取最近的实际录像帧，显示请求步、录像步及差值，**不会插值或伪造缺失帧**。修改后重新执行会保存新的分析版本。

In [ ]:
display(Video(str(FAILURE_EVALUATION / selected["video"]), embed=True))
KEYFRAME_STEPS = [0, selected["steps"] // 2, selected["steps"]]  # 改成想查看的控制步
FIRST_ERROR_STEP = None  # 学生观察后填整数，例如20；None表示尚未标记


In [ ]:
from manipulation_lab.reporting import failure_evidence
ANALYSIS = next_stage(RUN, "analysis")
ANALYSIS.mkdir()
facts = failure_evidence(FAILURE_EVALUATION, EPISODE, ANALYSIS,
                         requested_steps=KEYFRAME_STEPS, first_error_step=FIRST_ERROR_STEP)
display(Image(filename=str(ANALYSIS / "failure_signals.png")))
for frame in facts["keyframes"]:
    print("请求控制步", frame["requested_control_step"], "实际录像步", frame["recorded_control_step"],
          "差值", frame["offset_steps"], "视频帧索引", frame["video_frame_index"])
    display(Image(filename=str(ANALYSIS / frame["file"])))


填写失败阶段（允许 unknown）、第一次可见错误、可能原因和验证改进的方法。红线只是学生标注，不是自动推断的故障原因。未填写时报告保持 DRAFT（草稿）。

In [ ]:
FAILURE_STAGE = ""
FIRST_OBSERVABLE_ERROR = ""
HYPOTHESIZED_CAUSE = ""
IMPROVEMENT = ""
write_json(ANALYSIS / "failure_analysis.json", {
    "episode": EPISODE, "evidence_directory": os.path.relpath(FAILURE_EVALUATION, ANALYSIS),
    "evidence_policy_role": results["policy_role"], "evidence_checkpoint_sha256": results["checkpoint_sha256"],
    "stage": FAILURE_STAGE, "first_observable_error": FIRST_OBSERVABLE_ERROR,
    "first_error_control_step": FIRST_ERROR_STEP,
    "hypothesized_cause": HYPOTHESIZED_CAUSE, "improvement": IMPROVEMENT})
REPORT = next_stage(RUN, "report")
command("generate_report.py", "--run", RUN, "--output", REPORT)
from manipulation_lab.reporting import notebook_report_markdown
display(Markdown(notebook_report_markdown(REPORT / "report.md")))
print("提交前填写答案并整理为1–2页；两学习率对照、图和原始记录作为附件。")


提交包含：两组模型的真实训练曲线与checkpoint、同条件对照表、逐案例配对CSV、位置图、人工错误标记与关键帧。评分按理解与证据20%/25%/30%/25%，不以成功率高低排名。